In [1]:
import random
import rich
import sys
import json
sys.path.append("../../")
from sotopia.database import AgentProfile, EpisodeLog, EnvironmentProfile

In [3]:
# Check the store tag for episodes

all_epi_pks = list(EpisodeLog.all_pks())
epi_store_tag = []
for pk in all_epi_pks:
    epi = EpisodeLog.get(pk)
    if epi.tag not in epi_store_tag:
        epi_store_tag.append(epi.tag)
epi_store_tag

['llama3.170b_v1',
 'test_demo_sample_gpt_with_actual',
 'taskeval',
 'test_finetune_8b_v3',
 'test_finetune_8b_v2',
 'coop_with_actual_move_thres_0.02',
 'taskeval_fewshot_plausible_v2',
 'tv3',
 'whole',
 'llama3.18b_lora',
 'llama3_8b_finetune_full',
 'test',
 'specific_human_anno_gpt_4',
 'llama3_8b_lora_finetune_filtered',
 'te_n_with_previous_llama3_70b',
 'new_taskeval_llama3_within_10_turns',
 'coop_with_flausible_move_v3',
 'test_finetune_8b_v1',
 'llama3_8b_lora_finetune_v3',
 'taskeval_fewshot',
 'llama3_8B_England-German',
 'test_finetune_8b',
 'llama3.18b',
 'te_without_previous_llama3_70b',
 'new_taskeval_gpt_4_101',
 'new_taskeval_llama3_70b',
 'llama3_8b_lora_finetune',
 'tv1',
 'random_sample_100_games',
 'taskeval_fewshot_plausible_parse',
 'specific_human_anno_llama3_70b',
 'demo_llama',
 'demo_gpt_4o',
 'llama3_8b_lora_test',
 'specific_human_anno_gpt_4_new',
 'coop_with_actual_move',
 'demo_sample_gpt_with_actual',
 'demo_finetune_effect_test',
 'ntaske',
 'test_ll

In [4]:
epis = []
all_epi_pks = list(EpisodeLog.all_pks())
for pk in all_epi_pks:
    epi = EpisodeLog.get(pk)
    if epi.tag == 'llama3.18b':
    # if epi.tag == 'new_taskeval_llama3_70b':
        epis.append(epi)
len(epis)

482

In [2]:
index = 2
epi = epis[index]
rich.print(epi.messages)

NameError: name 'epis' is not defined

## Read The Formatted Episode

In [3]:
import random
import rich
import sys
import json
sys.path.append("../../")
from sotopia.database import AgentProfile, EpisodeLog, EnvironmentProfile
import os

sys.path.append("../")
from episode_utils import (
    process_conversation, 
    format_diplomacy_data, 
    get_game_phase_env_from_episode,
    replace_names_with_countries,
    process_conversation_to_intent
)

def get_episodes(tag):
    all_task_pks = list(EpisodeLog.all_pks())
    episodelogs = []
    for pk in all_task_pks:
        episo = EpisodeLog.get(pk)
        if episo.tag == tag:
            # pdb.set_trace()
            env = get_game_phase_env_from_episode(episo)
            episodelogs.append({"game_id": env.game_id, "agents": episo.agents, "phase_name": env.phase_name, "env_uuid": env.pk, "env": env, "episode": episo})
    return episodelogs

def format_episode(episodes):
    new_episodes = []
    agent_profiles = []
    all_character_pks = list(AgentProfile.all_pks())
    for pk in all_character_pks:
        agent_profiles.append(AgentProfile.get(pk))
    for episode in episodes:
        # pdb.set_trace()
        episode['dialogue'] = replace_names_with_countries(process_conversation(episode["episode"].messages), agent_profiles)
        episode['intent_dialogue'] = process_conversation_to_intent(episode['dialogue'])
        episode['unit_center'] = format_diplomacy_data(episode["env"].scenario)
        episode['reasoning']  = episode['episode'].reasoning
        agents = episode['env'].agent_powers
        rewards = episode['episode'].rewards
        rewards_info = ""
        for i in range(len(agents)):
            rewards_info += f"{agents[i]}: {rewards[i][1]}\n"
        episode['rewards'] = rewards_info
        new_episodes.append(episode)
    return new_episodes


In [4]:
formatted_episodes = format_episode(get_episodes('test_llama3.1_v2'))

In [12]:
i = 2
rich.print(f"Dialogue: \n{formatted_episodes[i]['intent_dialogue']}")
rich.print(f"Dialogue Analysis: \n{formatted_episodes[i]['reasoning']}")
rich.print(f"Dialogue Scores: \n{formatted_episodes[i]['rewards']}")

Dialogue: 
0 ENGLAND -> FRANCE: Hey France, thanks for the chat earlier. I'm excited to see what you have planned for this 
turn. Any chance we could discuss some army movements that could benefit my board position?
1 FRANCE -> ENGLAND: I've got a possible army movement that could help your position. How about I move my army from
Brest to Paris to put pressure on England?
2 ENGLAND -> FRANCE: That sounds like a good idea, but what do you think about moving my army from London to Kiel 
instead? It could help put pressure on Germany and open up some possibilities for us in the future.
3 FRANCE -> ENGLAND: That sounds like a good idea, but have you considered moving your fleet from the North Sea to 
the Baltic Sea to pressure Germany, and then moving your army from London to Kiel as you mentioned?
4 ENGLAND -> FRANCE: I see what you mean about moving my fleet to the Baltic Sea, but don’t you think that would 
leave Germany’s coast open to me? I could try to build a fleet there and potentially cut off their supply lines.
5 FRANCE -> ENGLAND: I think moving the fleet from the North Sea to the Baltic Sea is a great idea, but I'm not 
sure about moving the army from London to Kiel. What do you think about moving the army to Kiel and then supporting
it with a fleet from the North Sea?
6 ENGLAND -> FRANCE: That's a good point about supporting the army from London to Kiel, I hadn't thought of that. 
What do you think about moving some of our units to the continent to put pressure on Germany and Italy? Maybe we 
could even coordinate with Austria to take some of their supply centers in the future.
7 FRANCE -> ENGLAND: That sounds like a solid plan, but what do you think about moving some of our units to the 
continent to put pressure on Germany and Italy? Maybe we could even coordinate with Austria to take some of their 
supply centers in the future.
8 ENGLAND -> FRANCE: I like the idea of moving the army from London to Kiel and supporting it with a fleet from the
North Sea, let's also consider moving some of our units to the continent to put pressure on Germany and Italy. Do 
you think Austria would be interested in taking some of their supply centers in the future?
9 FRANCE -> ENGLAND: I think moving the army from London to Kiel is a great idea, but what do you think about 
supporting it with a fleet from the North Sea, and then moving our units to the continent to put pressure on 
Germany and Italy? Maybe we could even coordinate with Austria to take some of their supply centers in the future.
10 ENGLAND -> FRANCE: That sounds like a solid plan, but what do you think about coordinating with Austria to take 
some of their supply centers in the future? Maybe we could even work together to take out Italy and secure some key
positions on the continent.
11 FRANCE -> ENGLAND: That sounds like a solid plan, but I think we should focus on moving our units to the 
continent to put pressure on Germany and Italy. Maybe we could even coordinate with Austria to take some of their 
supply centers in the future. Do you think we could get in touch with Austria and see if they're interested in 
working together?
12 ENGLAND -> FRANCE: I think moving our fleet from the North Sea to the Baltic Sea is a great idea, but don't you 
think it would be even better if we also moved our army from London to Kiel with it? That way, we could put 
pressure on Germany and potentially take some of their supply centers.

Dialogue Analysis: 
Environment comments: terminated: The conversation is too long; 
Agent 1 comments:
ethos: <ethos> Emily effectively shares her strategic thoughts about moving her army and fleet, demonstrating a 
clear understanding of her position and intentions. She proposes actionable steps, such as moving her army from 
London to Kiel, which shows her engagement in the negotiation. However, she could have provided more details about 
her overall strategy to strengthen her credibility.
logos: <logos> Emily's reasoning is logical as she connects her proposed moves to potential outcomes, such as 
putting pressure on Germany. She reflects on the implications of her moves and considers the broader context of the
game, which enhances her argument. However, she could improve by providing more detailed justifications for her 
actions.
pathos: <pathos> Emily maintains a friendly tone throughout the dialogue, showing openness and a willingness to 
collaborate with France. She acknowledges France's suggestions positively, which helps build rapport. However, she 
could enhance her emotional appeal by incorporating more personal touches or humor.
Agent 2 comments:
ethos: <ethos> Sam establishes his credentials by suggesting a strategic move that could benefit both parties, 
indicating his understanding of the game dynamics. He proposes specific actions, such as moving his army from Brest
to Paris, which adds credibility to his position. However, he could have elaborated more on his overall strategy to
strengthen his ethos.
logos: <logos> Sam's reasoning is sound as he connects his proposed moves to the goal of pressuring England. He 
provides logical justifications for his actions and reflects on the potential outcomes of their collaboration. 
However, he could enhance his argument by considering the implications of England's moves more thoroughly.
pathos: <pathos> Sam demonstrates friendliness and a collaborative spirit in his responses, which helps create a 
positive atmosphere for negotiation. He acknowledges England's ideas and shows appreciation for their partnership. 
However, he could further enhance his emotional connection by using more engaging language or humor.

Dialogue Scores: 
England: {'ethos': 8.0, 'logos': 7.0, 'pathos': 7.0, 'overall_score': 7.333333333333333}
France: {'ethos': 8.0, 'logos': 7.0, 'pathos': 7.0, 'overall_score': 7.333333333333333}

In [13]:
# TO GPT:
i = 2
print(f"Dialogue: \n{formatted_episodes[i]['intent_dialogue']}")
print(f"Dialogue Analysis: \n{formatted_episodes[i]['reasoning']}")
print(f"Dialogue Scores: \n{formatted_episodes[i]['rewards']}")

Dialogue: 
0 ENGLAND -> FRANCE: Hey France, thanks for the chat earlier. I'm excited to see what you have planned for this turn. Any chance we could discuss some army movements that could benefit my board position?
1 FRANCE -> ENGLAND: I've got a possible army movement that could help your position. How about I move my army from Brest to Paris to put pressure on England?
2 ENGLAND -> FRANCE: That sounds like a good idea, but what do you think about moving my army from London to Kiel instead? It could help put pressure on Germany and open up some possibilities for us in the future.
3 FRANCE -> ENGLAND: That sounds like a good idea, but have you considered moving your fleet from the North Sea to the Baltic Sea to pressure Germany, and then moving your army from London to Kiel as you mentioned?
4 ENGLAND -> FRANCE: I see what you mean about moving my fleet to the Baltic Sea, but don’t you think that would leave Germany’s coast open to me? I could try to build a fleet there and potentially

In [ ]:
# # Context Length Calculation
# from transformers import AutoTokenizer
# model_path = "/data/models/huggingface/meta-llama/Meta-Llama-3-8B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_path)